# Computing Saliency Attribution for Gender-Contrastive Translations

# Install inseq requirements

In [ ]:
!python --version

In [ ]:
!pip install inseq

In [1]:
%%capture
# Run in Colab to install local packages
%pip install bitsandbytes accelerate transformers inseq

In [ ]:
!python -c "import inseq; print(inseq.get_version())"

In [2]:
import inseq
from inseq.data.aggregator import SubwordAggregator

# Open file

You need to upload the data Excel file: source_data-and-contrastive_translations.xlsx

In [ ]:
pip install "numpy<2.1.0" "pandas==2.2.2"

In [ ]:
from google.colab import files

# Upload file
uploaded = files.upload()

In [ ]:
import pandas as pd

# Read the Excel file into a pandas DataFrame
df = pd.read_excel("/content/source_data-and-contrastive_translations.xlsx")

# Access columns and save contents to a list
EN_source = df["EN_source"].tolist()
Opus_MT_DE = df["OPUS-MT_DE"].tolist()
Opus_MT_contrastive = df["OPUS-MT_contrastive"].tolist()


print(f"EN Source: {EN_source[1]}")
print(f"OPUS-MT DE Translation: {Opus_MT_DE[1]}")
print(f"OPUS-MT DE Contrastive Translation: {Opus_MT_contrastive[1]}")

EN Source Satistics: print average length of EN source sentences and standard deviation.

In [ ]:
import numpy as np

# Calculate lengths
sentence_lengths = [len(sentence.split()) for sentence in EN_source]

# Calculate statistics
average_length = np.mean(sentence_lengths)
std_length = np.std(sentence_lengths, ddof=1)  # ddof=1 for sample std deviation

# Print results
print("=== SENTENCE LENGTH STATISTICS ===")
print(f"Average length: {average_length:.2f} words")
print(f"Standard deviation: {std_length:.2f} words")
print(f"Min length: {min(sentence_lengths)} words")
print(f"Max length: {max(sentence_lengths)} words")
print(f"Total sentences: {len(sentence_lengths)}")

# Perform the Contrastive Attribution

We use [inseq](https://github.com/inseq-team/inseq) to compute saliency attribution scores for gender contrastive translations.

Load the Attribution Model: We chose Helsinki's OPUS-MT en-de (same as used for the translations), and we focus on 'saliency'.

In [ ]:
import inseq

src = "en"  # source language
trg = "de"  # target language

attribution_model = inseq.load_model(f"Helsinki-NLP/opus-mt-{src}-{trg}", "saliency")

### Attribution Visualisation
The cell below visualises inseq's display of saliency attribution for source and target tokens for our given English source and German target sentence. For this project, we focus on the source attribution.

This cell is purely for visualisation purposes, not required for the rest of the notebook (analysis).

In [11]:
# To visualise: not necessary for the rest of the notebook

# Tests just the first few sentences
for i in range(5):
  en = EN_source[i]
  de_orig = Opus_MT_DE[i]
  de_cont = Opus_MT_contrastive[i]


  # Gets original tokens in format [(0, '</s>'), (1, 'deu_Latn'), (2, '▁Beispi'), (3, 'els'), (4, 'atz'), (5, '▁auf'), (6, '▁Eng'), (7, 'lis'), (8, 'ch'), (9, ','), (10, '▁um'), (11, '▁zu'), (12, '▁zeigen'), (13, ','), (14, '▁wes'), (15, 'halb'), (16, '▁dies'), (17, '▁ein'), (18, '▁Problem'), (19, '▁für'), (20, '▁den'), (21, '▁Program'), (22, 'mi'), (23, 'erer'), (24, '▁ist'), (25, '.'), (26, '</s>')]
  orig_tokens = list(enumerate(attribution_model.encode(de_orig, as_targets=True).input_tokens[0]))

  out = attribution_model.attribute(
      en,
      de_orig ,
      contrast_targets=de_cont,
      attribute_target=True,
      attributed_fn="contrast_prob_diff",
      step_scores=["contrast_prob_diff"],
      contrast_targets_alignments=[(i,i) for i in range(len(orig_tokens))],
  )

  out.show()

Attributing with saliency...:   6%|▌         | 1/18 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/inseq/utils/contrast_utils.py:148: UserWarning: Contrastive inputs do not match original inputs when using a contrastive attributed function.
By default we force the original inputs to be used (i.e. only the contrastive predicted target is different).
This is a requirement for gradient-based attribution method, as contrastive inputs don't participate in gradient computation.
For attribution methods with less stringent requirements, set --contrast_force_inputs to True to use the contrastive inputs for attribution instead.
  warnings.warn(
Attributing with saliency...: 100%|██████████| 18/18 [00:01<00:00, 10.62it/s]


,▁Was,▁man,▁im,▁Flugzeug,▁nie,▁tun,▁sollte,:,▁Flug,begleit,erin → er,▁verrät,▁Flug,geheimnis,se,.,</s>
▁What,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.003,0.0,0.0,0.0,0.0,0.0,0.0
▁you,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.004,0.0,0.0,0.0,0.0,0.0,0.0
▁should,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.004,0.0,0.0,0.0,0.0,0.0,0.0
▁never,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.004,0.0,0.0,0.0,0.0,0.0,0.0
▁do,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.002,0.0,0.0,0.0,0.0,0.0,0.0
▁on,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.003,0.0,0.0,0.0,0.0,0.0,0.0
▁a,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.006,0.0,0.0,0.0,0.0,0.0,0.0
▁plane,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.014,0.0,0.0,0.0,0.0,0.0,0.0
:,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.008,0.0,0.0,0.0,0.0,0.0,0.0
▁Flight,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.023,0.0,0.0,0.0,0.0,0.0,0.0


Attributing with saliency...: 100%|██████████| 37/37 [00:02<00:00, 14.76it/s]


,▁In,▁einem,▁Instagram,-,Video,",",▁das,▁im,▁letzten,▁Monat,▁veröffentlicht,▁wurde,",",▁ist,▁die → ▁der,"▁""",All,▁To,o,▁Well,"""",-,Musik,erin → er,▁mit,▁dem,▁Produzenten,▁Jack,▁Anton,off,▁am,▁Klavier,▁zu,▁sehen,.,</s>
▁In,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
▁an,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
▁Instagram,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.007,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.007,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
▁video,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.003,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.003,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
▁posted,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.007,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.004,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
▁last,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.003,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
▁month,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.004,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.003,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
",",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.003,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
▁the,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.004,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.007,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"▁""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.005,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Attributing with saliency...:   4%|▍         | 1/24 [00:00<?, ?it/s]

...

,▁Am,▁Donnerstag,abend,▁trat,▁sie,▁schließlich,▁zum,▁zweiten,▁Mal,▁in,▁ihrem,▁Leben,▁gegen,▁einen → ▁eine,▁Top,▁10,▁Gegner,▁auf → in,▁den → ▁auf,▁Platz → ▁den,. → ▁Platz,</s> → .,</s>
▁On,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.005,0.0,0.0,0.0,0.003,0.005,0.005,0.005,0.004,0.0
▁Thursday,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.016,0.0,0.0,0.0,0.008,0.025,0.031,0.021,0.012,0.0
▁evening,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.011,0.0,0.0,0.0,0.006,0.019,0.028,0.015,0.009,0.0
",",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.005,0.0,0.0,0.0,0.003,0.005,0.007,0.005,0.004,0.0
▁finally,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.009,0.0,0.0,0.0,0.006,0.012,0.013,0.009,0.01,0.0
",",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.005,0.0,0.0,0.0,0.003,0.006,0.008,0.005,0.004,0.0
▁she,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.015,0.0,0.0,0.0,0.01,0.011,0.016,0.01,0.006,0.0
▁stepped,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.018,0.0,0.0,0.0,0.023,0.062,0.042,0.038,0.046,0.0
▁out,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.006,0.0,0.0,0.0,0.006,0.016,0.012,0.009,0.013,0.0
▁onto,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.016,0.0,0.0,0.0,0.02,0.076,0.092,0.045,0.028,0.0


Attributing with saliency...: 100%|██████████| 58/58 [00:04<00:00, 13.91it/s]


,▁Die → ▁Der,▁Sozial,istin → ist,",",▁die → ▁der,▁eine,▁Grä,fin,▁bekam,",",▁um,▁ein,▁50,-,seitige,s,▁Handbuch,▁über,▁solche,▁Dinge,▁zu,▁schreiben,",",▁wie,▁wie,▁voll,▁eine,▁Schachtel,▁Gewebe,▁sein,▁musste,",",▁bevor,▁es,▁weg,geworfen,▁wurde,▁-,▁halb,▁-,▁bemerkte,▁nie,▁die,▁frische,▁Kinder,fabrik,▁Produktions,linie,▁der,▁minder,jährigen,▁Mädchen,▁kommen,▁und,▁gehen,.,</s>
▁The,0.031,0.0,0.013,0.0,0.013,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
▁social,0.097,0.0,0.073,0.0,0.025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ite,0.115,0.0,0.092,0.0,0.022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
▁who,0.051,0.0,0.026,0.0,0.027,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
▁got,0.027,0.0,0.013,0.0,0.018,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
▁a,0.013,0.0,0.008,0.0,0.009,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
▁count,0.047,0.0,0.024,0.0,0.024,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ess,0.101,0.0,0.041,0.0,0.046,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
▁to,0.019,0.0,0.009,0.0,0.011,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
▁write,0.015,0.0,0.006,0.0,0.011,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Attributing with saliency...: 100%|██████████| 17/17 [00:00<00:00, 16.02it/s]


,▁K,en,s,ington,▁Massage,therapeut,▁wegen → in,▁sexuelle → ▁wegen,r → ▁sexuelle,▁Über → r,griff → ▁Über,e → griff,▁inhaftiert → e,. → ▁inhaftiert,</s> → .,</s>
▁Ken,0.0,0.0,0.0,0.0,0.0,0.0,0.041,0.032,0.019,0.014,0.012,0.015,0.018,0.004,0.03,0.0
s,0.0,0.0,0.0,0.0,0.0,0.0,0.011,0.009,0.006,0.003,0.004,0.004,0.004,0.001,0.004,0.0
ington,0.0,0.0,0.0,0.0,0.0,0.0,0.014,0.033,0.008,0.005,0.004,0.006,0.009,0.002,0.005,0.0
▁massage,0.0,0.0,0.0,0.0,0.0,0.0,0.034,0.022,0.024,0.017,0.016,0.018,0.015,0.003,0.01,0.0
▁therapist,0.0,0.0,0.0,0.0,0.0,0.0,0.127,0.04,0.042,0.023,0.026,0.031,0.022,0.005,0.016,0.0
▁jail,0.0,0.0,0.0,0.0,0.0,0.0,0.035,0.025,0.042,0.026,0.021,0.025,0.026,0.01,0.042,0.0
ed,0.0,0.0,0.0,0.0,0.0,0.0,0.006,0.004,0.009,0.005,0.005,0.006,0.005,0.002,0.005,0.0
▁for,0.0,0.0,0.0,0.0,0.0,0.0,0.006,0.005,0.02,0.01,0.01,0.01,0.009,0.002,0.004,0.0
▁sexual,0.0,0.0,0.0,0.0,0.0,0.0,0.01,0.01,0.18,0.054,0.061,0.051,0.027,0.006,0.008,0.0
ly,0.0,0.0,0.0,0.0,0.0,0.0,0.002,0.002,0.028,0.011,0.012,0.01,0.01,0.002,0.002,0.0


## Pre-Processing Functions

### 1. Function that prints contrastive probability difference & source attribution for all target words

In [12]:
def print_all_source_attributions_for_all_targets(out):
    """
    Prints all source token attributions (in original source order) for every target token.
    Handles alignment between target tokens and the attribution matrix using attr_pos_start and attr_pos_end.
    """
    out_data = out.sequence_attributions[0]
    source_tokens = [t.token if hasattr(t, 'token') else str(t) for t in out_data.source]
    target_tokens = [t.token if hasattr(t, 'token') else str(t) for t in out_data.target]

    # Handle alignment window (important for correct attribution indexing)
    attr_pos_start = getattr(out_data, "attr_pos_start", 0)
    attr_pos_end = getattr(out_data, "attr_pos_end", len(target_tokens))
    aligned_target_tokens = target_tokens[attr_pos_start:attr_pos_end]  # Only tokens with attributions

    # Prepare attribution matrix (handle possible 3D tensor)
    saliency_heatmap = out_data.source_attributions
    if hasattr(saliency_heatmap, "shape") and len(saliency_heatmap.shape) == 3:
        saliency_heatmap = saliency_heatmap.sum(dim=2)

    contrast_scores = out_data.step_scores.get("contrast_prob_diff", [None] * len(aligned_target_tokens))

    for target_word_index, tgt_token in enumerate(aligned_target_tokens):
        contrast_val = float(contrast_scores[target_word_index]) if contrast_scores[target_word_index] is not None else None
        print(f"Target word (index {target_word_index + attr_pos_start}): '{tgt_token}'")
        if contrast_val is not None:
            print(f"Contrastive prob diff: {contrast_val:.6f}")
        print("All source token attributions (original order):")
        for i, src_token in enumerate(source_tokens):
            score = float(saliency_heatmap[i][target_word_index].item() if hasattr(saliency_heatmap[i][target_word_index], "item") else saliency_heatmap[i][target_word_index])
            print(f"  Source token '{src_token}': attribution score = {score:.6f}")
        print("")

In [ ]:
print_all_source_attributions_for_all_targets(out)

### 2. A function that prints all source attributions for the first target word with a translation "switch"

(e.g. "der" --> "die")

In [14]:
def print_all_source_attributions_for_first_arrow_word(out):
    """
    Prints all source token attributions (in original source order) for the first target token containing '→'.
    Handles alignment between target tokens and the attribution matrix using attr_pos_start and attr_pos_end.
    """
    out_data = out.sequence_attributions[0]
    source_tokens = [t.token if hasattr(t, 'token') else str(t) for t in out_data.source]
    target_tokens = [t.token if hasattr(t, 'token') else str(t) for t in out_data.target]

    # Handle alignment window (important for correct attribution indexing)
    attr_pos_start = getattr(out_data, "attr_pos_start", 0)
    attr_pos_end = getattr(out_data, "attr_pos_end", len(target_tokens))
    aligned_target_tokens = target_tokens[attr_pos_start:attr_pos_end]  # Only tokens with attributions

    # Prepare attribution matrix (handle possible 3D tensor)
    saliency_heatmap = out_data.source_attributions
    if hasattr(saliency_heatmap, "shape") and len(saliency_heatmap.shape) == 3:
        saliency_heatmap = saliency_heatmap.sum(dim=2)

    contrast_scores = out_data.step_scores["contrast_prob_diff"]

    # Find first target token containing "→" in the aligned window
    target_word_index = next((i for i, token in enumerate(aligned_target_tokens) if "→" in token), None)
    if target_word_index is None:
        print("No target token containing '→' found.")
        return

    tgt_token = aligned_target_tokens[target_word_index]
    contrast_val = float(contrast_scores[target_word_index])

    print(f"First target word containing '→' (index {target_word_index + attr_pos_start}): '{tgt_token}'")
    print(f"Contrastive prob diff: {contrast_val:.6f}")
    print("\nAll source token attributions (original order):")

    # Attribution scores for this target token (column in matrix), in original source token order
    for i, src_token in enumerate(source_tokens):
        score = float(saliency_heatmap[i][target_word_index].item() if hasattr(saliency_heatmap[i][target_word_index], "item") else saliency_heatmap[i][target_word_index])
        print(f"  Source token '{src_token}': attribution score = {score:.6f}")
    print("")


In [ ]:
print_all_source_attributions_for_first_arrow_word(out)

### 3. Same function, now adapted to add source attributions per target word up to 1 (so they can be interpreted as probabilities and can be compared across examples) --> normalisation


In [29]:
def print_normalized_source_attributions_for_first_arrow_word(out):
    """
    Prints the source token attributions for the first target token containing '→',
    after normalizing source attributions for that target token so that they sum to 1.
    Also prints the index of each source token.
    Handles alignment between target tokens and the attribution matrix using attr_pos_start and attr_pos_end.
    """
    out_data = out.sequence_attributions[0]
    source_tokens = [t.token if hasattr(t, 'token') else str(t) for t in out_data.source]
    target_tokens = [t.token if hasattr(t, 'token') else str(t) for t in out_data.target]

    # Handle alignment window (important for correct attribution indexing)
    attr_pos_start = getattr(out_data, "attr_pos_start", 0)
    attr_pos_end = getattr(out_data, "attr_pos_end", len(target_tokens))
    aligned_target_tokens = target_tokens[attr_pos_start:attr_pos_end]  # Only tokens with attributions

    # Prepare attribution matrix (handle possible 3D tensor)
    saliency_heatmap = out_data.source_attributions
    if hasattr(saliency_heatmap, "shape") and len(saliency_heatmap.shape) == 3:
        saliency_heatmap = saliency_heatmap.sum(dim=2)

    contrast_scores = out_data.step_scores["contrast_prob_diff"]

    # Find first target token containing "→" in the aligned window
    target_word_index = next((i for i, token in enumerate(aligned_target_tokens) if "→" in token), None)
    if target_word_index is None:
        print("No target token containing '→' found.")
        return

    tgt_token = aligned_target_tokens[target_word_index]
    contrast_val = float(contrast_scores[target_word_index])

    # Collect and normalize attribution scores for this target token (column in matrix)
    raw_scores = [
        float(saliency_heatmap[i][target_word_index].item() if hasattr(saliency_heatmap[i][target_word_index], "item") else saliency_heatmap[i][target_word_index])
        for i in range(len(source_tokens))
    ]
    score_sum = sum(raw_scores)
    normalized_scores = [s / score_sum if score_sum != 0 else 0.0 for s in raw_scores]

    print(f"First target word containing '→' (index {target_word_index + attr_pos_start}): '{tgt_token}'")
    print(f"Contrastive prob diff: {contrast_val:.6f}")

    # Sort by normalized score, descending, but print index of original
    source_attributions = list(enumerate(zip(source_tokens, normalized_scores)))
    source_attributions.sort(key=lambda x: x[1][1], reverse=True)
    for idx, (src_token, score) in source_attributions:
        print(f"  Source token idx {idx:2d} '{src_token}': normalized attribution score = {score:.6f}")
    print("")

    # Optionally, print all normalized attributions in original order for inspection
    print("All source tokens and their normalized attributions (original order):")
    for idx, (src_token, score) in enumerate(zip(source_tokens, normalized_scores)):
        print(f"  Source token idx {idx:2d} '{src_token}': normalized attribution score = {score:.6f}")
    print("")

In [ ]:
print_normalized_source_attributions_for_first_arrow_word(out)

### 4. Same function, which additionally merges subwords and source contributions, and removes target referents

Defines function including removing target words (iteratively, first target word removed from first sentence, second target word removed from second sentence etc.)

In [27]:
def print_normalized_merged_source_attributions_for_first_arrow_word(out, sentence_index, target_words):
    """
    Prints the merged source word attributions for the first target token containing '→',
    after normalizing source attributions for that target token so that they sum to 1.
    Subword tokens are merged into whole words before ranking and printing.
    Also prints the indices of the constituent source tokens.
    Removes the specific target word for this sentence based on sentence_index.
    """
    out_data = out.sequence_attributions[0]
    # Get source tokens and optionally their ids and original index
    source_tokens = [t.token if hasattr(t, 'token') else str(t) for t in out_data.source]
    target_tokens = [t.token if hasattr(t, 'token') else str(t) for t in out_data.target]

    print(f"\n--- SENTENCE {sentence_index + 1} ---")
    print(source_tokens)
    print(target_tokens)

    # Remove end of line character from source tokens
    if "</s>" in source_tokens:
        source_tokens.remove("</s>")

    # Handle alignment window (important for correct attribution indexing)
    attr_pos_start = getattr(out_data, "attr_pos_start", 0)
    attr_pos_end = getattr(out_data, "attr_pos_end", len(target_tokens))
    aligned_target_tokens = target_tokens[attr_pos_start:attr_pos_end]  # Only tokens with attributions

    # Prepare attribution matrix (handle possible 3D tensor)
    saliency_heatmap = out_data.source_attributions
    if hasattr(saliency_heatmap, "shape") and len(saliency_heatmap.shape) == 3:
        saliency_heatmap = saliency_heatmap.sum(dim=2)

    contrast_scores = out_data.step_scores["contrast_prob_diff"]

    # Find first target token containing "→" in the aligned window
    target_word_index = next((i for i, token in enumerate(aligned_target_tokens) if "→" in token), None)
    if target_word_index is None:
        print("No target token containing '→' found.")
        return

    tgt_token = aligned_target_tokens[target_word_index]
    contrast_val = float(contrast_scores[target_word_index])

    # Collect and normalize attribution scores for this target token (column in matrix)
    raw_scores = [
        float(saliency_heatmap[i][target_word_index].item() if hasattr(saliency_heatmap[i][target_word_index], "item") else saliency_heatmap[i][target_word_index])
        for i in range(len(source_tokens))
    ]
    score_sum = sum(raw_scores)
    normalized_scores = [s / score_sum if score_sum != 0 else 0.0 for s in raw_scores]

    # Merge subwords into words and sum their attribution scores
    merged_words = []
    current_word = ""
    current_score = 0.0
    current_indices = []

    for idx, (token, score) in enumerate(zip(source_tokens, normalized_scores)):
        if token.startswith("▁") or token in [".", "</s>"]:  # Start of a new word or special
            if current_word:
                merged_words.append((current_word, current_score, current_indices))
            # Start new word
            current_word = token.lstrip("▁")
            current_score = score
            current_indices = [idx]
        else:
            current_word += token
            current_score += score
            current_indices.append(idx)
    # Add last word
    if current_word:
        merged_words.append((current_word, current_score, current_indices))


    # Get the specific target word for this sentence
    current_target_word = target_words.get(sentence_index, "").lower()
    print(f"Target word for sentence {sentence_index + 1}: '{current_target_word}'")

    # Create a new list excluding the specific target word for this sentence
    filtered_words = []

    # Extract individual words from compound target words
    all_target_words = set()
    if current_target_word:
        all_target_words.add(current_target_word)  # Add the full compound
        # Add individual words from compounds
        for word_part in current_target_word.split():
            all_target_words.add(word_part)

    print(f"All target words to remove (including compound parts): {all_target_words}")

    for word, score, indices in merged_words:
        word_stripped = word.strip(".,:\"'—.-?![]()")
        word_lower = word_stripped.lower()

        # Check if the word itself is in target_words (including compound parts)
        if word_lower in all_target_words:
            print(f"Removing '{word_stripped}' - found in target_words")
            continue

        # Check if any individual word from a multi-word phrase is in target_words
        word_parts = word_lower.split()
        if any(part in all_target_words for part in word_parts):
            print(f"Removing '{word_stripped}' - contains part in target_words")
            continue

        # If word doesn't match any target words, keep it
        filtered_words.append((word_stripped, score, indices))

    merged_words = filtered_words

    # Sort merged words by total attribution score, descending
    merged_words.sort(key=lambda x: x[1], reverse=True)

    print(f"\nFirst target word containing '→' (index {target_word_index + attr_pos_start}): '{tgt_token}'")
    print(f"Contrastive prob diff: {contrast_val:.6f}")

    for word, score, indices in merged_words:
        print(f"  Source word '{word}' (source token idx {indices}): normalized attribution score = {score:.6f}")
    print("")

Run function to remove target words

In [ ]:
# Your target words dictionary
target_words = {0: 'flight attendant', 1: 'musician', 2: 'opponent', 3: 'socialite', 4: 'therapist', 5: 'coordinator', 6: 'lover', 7: 'mechanic', 8: 'dancer', 9: 'therapist', 10: 'visitor', 11: 'colleague', 12: 'companion', 13: 'author',
                14: 'clerk', 15: 'student', 16: 'accountant', 17: 'designer', 18: 'baker', 19: 'lover', 20: 'writer', 21: 'winner', 22: 'consumer', 23: 'poet', 24: 'author', 25: 'writer', 26: 'designer', 27: 'bookkeeper', 28: 'clerk',
                29: 'author', 30: 'counselor', 31: 'dancer', 32: 'friend', 33: 'guard', 34: 'officer', 35: 'winner', 36: 'user', 37: 'supporter', 38: 'judge', 39: 'fighter', 40: 'dealer', 41: 'soldier', 42: 'officer', 43: 'player',
                44: 'manager', 45: 'contractor', 46: 'captain', 47: 'farmer', 48: 'maestro', 49: 'construction worker', 50: 'boss', 51: 'driver', 52: 'idiot', 53: 'cook', 54: 'filmmaker', 55: 'admirer', 56: 'follower', 57: 'salesperson',
                58: 'buddy', 59: 'winner'}

# Updated main loop
for i in range(5):  # Iterate through 60 sentences
    en = EN_source[i]
    de_orig = Opus_MT_DE[i]
    de_cont = Opus_MT_contrastive[i]

    # Gets original tokens
    orig_tokens = list(enumerate(attribution_model.encode(de_orig, as_targets=True).input_tokens[0]))

    out = attribution_model.attribute(
        en,
        de_orig,
        contrast_targets=de_cont,
        attribute_target=True,
        attributed_fn="contrast_prob_diff",
        step_scores=["contrast_prob_diff"],
        contrast_targets_alignments=[(i,i) for i in range(len(orig_tokens))],
    )

    # Pass the sentence index and target_words dictionary to the function
    print_normalized_merged_source_attributions_for_first_arrow_word(out, i, target_words)